In [ ]:
import sys
import os


from tqdm import tqdm
import time


import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader


sys.path.append(os.path.abspath(".."))
from package.SingleCharTokenizer import * 
from package.TextDataset import * 
from package.GPT import * 


%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [141]:
#d_model=256, n_heads=4, n_layers=6, d_ff=1024, block_size=128
sliding_windows = 0.

context_window = 128
batch_size = 64

d_emb = 256
nb_heads = 4
d_k = d_emb // nb_heads

nb_layers = 6

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
#device = torch.device("cpu")
print(device)

mps


In [143]:
tokenizer = SingleCharTokenizer()
tokens = torch.tensor(tokenizer.load_tokens("tokens.tok"))
print(tokens.shape)


vocab_size = tokenizer.vocab_size
print("vocab_size : " , vocab_size)

dataset = TextDataset(tokens,context_window=context_window,sliding_windows=sliding_windows)
print("dataset_size : " ,len(dataset))


loader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True,
    drop_last=True,
    num_workers = 2
)


#X,Y=next(iter(loader))
#print(X.dtype)
#print(X.device)

torch.Size([11473626])
vocab_size :  65
dataset_size :  11473499


179273


In [145]:
gpt = GPT(vocab_size=vocab_size, context_window=context_window, d_emb=d_emb,nb_layers=nb_layers,nb_heads=nb_heads).to(device)


In [146]:
#gpt.save("model.w")
#gpt2 = GPT().load("model.w")
#gpt.state_dict()["embedding.E.weight"]
#print(gpt2.state_dict()['embedding.E.weight'])
print(next(gpt.parameters()).device)


mps:0


In [147]:
nb_param = 0 
for name, param in gpt.named_parameters():
    c= 1
    for i in param.size():
        c *= i
    nb_param += c

    print(f"{name}: {param.device} {param.size()} {c}")

print(f"Total number of parameters : {nb_param}")

embedding.E.weight: mps:0 torch.Size([65, 256]) 16640
embedding.P.weight: mps:0 torch.Size([128, 256]) 32768
transformer_blocks.0.norm1.weight: mps:0 torch.Size([256]) 256
transformer_blocks.0.norm1.bias: mps:0 torch.Size([256]) 256
transformer_blocks.0.norm2.weight: mps:0 torch.Size([256]) 256
transformer_blocks.0.norm2.bias: mps:0 torch.Size([256]) 256
transformer_blocks.0.attention.Wq.weight: mps:0 torch.Size([256, 256]) 65536
transformer_blocks.0.attention.Wk.weight: mps:0 torch.Size([256, 256]) 65536
transformer_blocks.0.attention.Wv.weight: mps:0 torch.Size([256, 256]) 65536
transformer_blocks.0.attention.Wo.weight: mps:0 torch.Size([256, 256]) 65536
transformer_blocks.0.feed_forward.NN.0.weight: mps:0 torch.Size([768, 256]) 196608
transformer_blocks.0.feed_forward.NN.0.bias: mps:0 torch.Size([768]) 768
transformer_blocks.0.feed_forward.NN.2.weight: mps:0 torch.Size([256, 768]) 196608
transformer_blocks.0.feed_forward.NN.2.bias: mps:0 torch.Size([256]) 256
transformer_blocks.1.no

In [148]:
for name, buf in gpt.named_buffers():
    print(name, buf.device)

embedding.position mps:0
transformer_blocks.0.attention.M mps:0
transformer_blocks.1.attention.M mps:0
transformer_blocks.2.attention.M mps:0
transformer_blocks.3.attention.M mps:0
transformer_blocks.4.attention.M mps:0
transformer_blocks.5.attention.M mps:0


In [ ]:
for param_tensor in gpt.state_dict():
    print(param_tensor, "\t", gpt.state_dict()[param_tensor].size())

embedding.position 	 torch.Size([128])
embedding.E.weight 	 torch.Size([65, 256])
embedding.P.weight 	 torch.Size([128, 256])
transformer_blocks.0.norm1.weight 	 torch.Size([256])
transformer_blocks.0.norm1.bias 	 torch.Size([256])
transformer_blocks.0.norm2.weight 	 torch.Size([256])
transformer_blocks.0.norm2.bias 	 torch.Size([256])
transformer_blocks.0.attention.M 	 torch.Size([128, 128])
transformer_blocks.0.attention.Wq.weight 	 torch.Size([256, 256])
transformer_blocks.0.attention.Wk.weight 	 torch.Size([256, 256])
transformer_blocks.0.attention.Wv.weight 	 torch.Size([256, 256])
transformer_blocks.0.attention.Wo.weight 	 torch.Size([256, 256])
transformer_blocks.0.feed_forward.NN.0.weight 	 torch.Size([768, 256])
transformer_blocks.0.feed_forward.NN.0.bias 	 torch.Size([768])
transformer_blocks.0.feed_forward.NN.2.weight 	 torch.Size([256, 768])
transformer_blocks.0.feed_forward.NN.2.bias 	 torch.Size([256])
transformer_blocks.1.norm1.weight 	 torch.Size([256])
transformer_bloc

In [150]:
print(len(loader))

179273


In [151]:
counter = 0

saving_frequency = 1000

nb_steps = 10000

optimizer = torch.optim.Adam(gpt.parameters(), lr=3e-4)


#for step,(x, y) in enumerate((loader)):
for step in tqdm(range(nb_steps)):
    x, y = next(iter(loader))
    x = x.to(device)
    y = y.to(device)

    if step%saving_frequency==0:
        filename = f"training_historic/{counter:04d}.w"
        gpt.save(filename)
        counter+=1

    optimizer.zero_grad(set_to_none=True)
    logits = gpt(x)
    loss = F.cross_entropy(logits.view(-1, vocab_size), y.view(-1))
    loss.backward()
    optimizer.step()

filename = f"training_historic/final.w"
gpt.save(filename)

100%|██████████| 10000/10000 [3:26:56<00:00,  1.24s/it] 


In [153]:
nb_steps = 20000
#for step,(x, y) in enumerate((loader)):
for step in tqdm(range(nb_steps)):
    x, y = next(iter(loader))
    x = x.to(device)
    y = y.to(device)

    if step%saving_frequency==0:
        filename = f"training_historic/{counter:04d}.w"
        gpt.save(filename)
        counter+=1

    optimizer.zero_grad(set_to_none=True)
    logits = gpt(x)
    loss = F.cross_entropy(logits.view(-1, vocab_size), y.view(-1))
    loss.backward()
    optimizer.step()

filename = f"training_historic/final3.w"
gpt.save(filename)

100%|██████████| 20000/20000 [4:46:23<00:00,  1.16it/s]  


gpu 38s

cpu 59s

gpu with saved M 36s

gpu with saved M and position 40s